# 06 — Train best config (manual hyperparameters, registry)

**Summary:** One final LoRA fine-tune with **hyperparameters you set** in this notebook (copy from your tuning results in **04–05**). Registers under `tuning_stage='best_extended'`.

**Prerequisites:** Notebook **03** (splits); complete hypertuning elsewhere, then paste winning values into **`EXTENDED_TRAIN_CONFIG`** and **`MAX_SEQ_LENGTH`** below.

**Recommended run order:** 03 → 04 → 05 → **08** (holdout eval for tuning models) → **06** (this notebook) → 07 → **09** (post-train holdout eval) → 10–12.

#### Colab / install

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install transformers peft accelerate bitsandbytes datasets trl cairosvg pillow scikit-learn lxml pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.0 MB/s eta 0:00:00


#### Imports + parameters

In [3]:
import sys
from pathlib import Path

import torch

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RUN_PROFILE_ID = 'run_1' #'default'
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID
MODELS_ROOT = WORKFLOW_ROOT / 'models'
EXPERIMENT_ROOT = WORKFLOW_ROOT / 'lora_tuning_workflow'

SEED = 42
VAL_FRAC = 0.005
TRAIN_FIRST_N =   # first N rows of train split; None = full train (shuffled split)

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'

# Paste from your best tuning row (e.g. round4_results.csv), then adjust.
MAX_SEQ_LENGTH = 2560

EXTENDED_TRAIN_CONFIG = {
    'learning_rate': 2e-4,
    'max_steps': 500,
    'warmup_ratio': 0.1,
    'lr_scheduler_type': 'cosine',
    'early_stopping_patience': 2,
    'early_stopping_min_delta': 0.0,
    'eval_steps': 50,
    'lora_r': 64,
    'lora_alpha': 128,
    'lora_dropout': 0.05,
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 4,
}

print('WORKFLOW_ROOT:', WORKFLOW_ROOT)
print('cuda:', torch.cuda.is_available())

WORKFLOW_ROOT: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/outputs/workflow_runs/run_1
cuda: True


#### Load data, train final extended run

In [4]:
from src.core.dataframe import choose_first_existing, train_val_split_df
from src.core.modeling_splits import load_train_val_pool
from src.core.runtime import cleanup_memory, set_seed
from src.training.lora.experiments import run_single_experiment_eval_loss_early_stop

set_seed(SEED)
cleanup_memory()

train_val_pool = load_train_val_pool(WORKFLOW_ROOT)
PROMPT_COL = choose_first_existing(train_val_pool, ['prompt', 'description', 'text'], 'pool')
SVG_COL = choose_first_existing(train_val_pool, ['svg', 'svg_code', 'target', 'label'], 'pool')
_shuffle = TRAIN_FIRST_N is None
train_df, val_df = train_val_split_df(
    train_val_pool, val_frac=VAL_FRAC, seed=SEED, shuffle=_shuffle
)
if TRAIN_FIRST_N is not None:
    train_df = train_df.iloc[: int(TRAIN_FIRST_N)].reset_index(drop=True)
    n_val = min(len(val_df), max(1, int(VAL_FRAC * TRAIN_FIRST_N)))
    val_df = val_df.sample(n=n_val, random_state=SEED).reset_index(drop=True)

cfg = dict(EXTENDED_TRAIN_CONFIG)
max_seq_length = int(MAX_SEQ_LENGTH)

run_name = f'best_extended_msl{max_seq_length}_seed{SEED}'
summary, _, run_dir, reg_id = run_single_experiment_eval_loss_early_stop(
    run_name=run_name,
    config=cfg,
    train_df=train_df,
    val_df=val_df,
    prompt_col=PROMPT_COL,
    svg_col=SVG_COL,
    model_id=MODEL_ID,
    max_seq_length=max_seq_length,
    root_dir=EXPERIMENT_ROOT,
    project_root=PROJECT_DIR,
    tuning_stage='best_extended',
    curriculum=False,
    seed=SEED,
    eval_steps=cfg.get('eval_steps'),
    notes='best extended (manual hparams)',
    models_root=MODELS_ROOT,
)

print('registry_model_id:', reg_id)
print('summary:', summary)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/49650 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
50,0.469708,0.420981
100,0.368616,0.378217
150,0.359751,0.358779
200,0.370451,0.347396
250,0.317486,0.339393
300,0.351670,0.331114
350,0.342108,0.325448
400,0.331633,0.321705
450,0.344214,0.320841
500,0.331707,0.320759


registry_model_id: 9c771d06b1014d22
summary: {'run_name': 'best_extended_msl2560_seed42', 'train_rows': 49650, 'val_rows': 250, 'learning_rate': 0.0002, 'lora_r': 32, 'lora_alpha': 64, 'lora_dropout': 0.05, 'per_device_train_batch_size': 4, 'gradient_accumulation_steps': 4, 'effective_batch_size': 16, 'max_steps': 500, 'warmup_ratio': 0.1, 'lr_scheduler_type': 'cosine', 'train_loss': 0.36432977056503296, 'train_runtime': 4102.7075, 'eval_loss': 0.32075902819633484, 'best_loss_checkpoint': '/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/outputs/workflow_runs/run_1/lora_tuning_workflow/best_extended_msl2560_seed42/checkpoint-500', 'svg_open_rate': nan, 'svg_close_rate': nan, 'xml_parse_rate': nan, 'render_rate': nan, 'avg_pred_char_len': nan, 'submission_valid_rate': nan, 'registry_model_id': '9c771d06b1014d22'}
